# RL-08 : Model-Based RL — Dyna-Q et planification

**Série** : Reinforcement Learning | **Notebook jumeau** : RL-08 | **Durée estimée** : 45–50 min

**Navigation** : [Version Python](RL-08-Dyna-Q-Planification-Python.ipynb) | [README de la série](README.md)

**Prérequis** : RL-05 (MDP et Q-Learning tabulaire), C# intermédiaire et .NET 9.

## Objectifs pédagogiques

- Distinguer apprentissage model-free et model-based.
- Construire un modèle tabulaire déterministe et l’échantillonner.
- Implémenter Q-Learning, Dyna-Q et Dyna-Q+ depuis zéro avec le BCL.
- Mesurer l’effet du nombre de mises à jour simulées à expérience réelle égale.
- Tester l’adaptation à un environnement changeant et évaluer des rollouts.

**Plan** : concepts et labyrinthe ; Q-Learning ; modèle tabulaire ; Dyna-Q ; Blocking Maze et Dyna-Q+ ; rollouts ; trois exercices ; synthèse.

Aucune bibliothèque RL ni service externe n’est requis : la source présente elle-même des algorithmes tabulaires didactiques, sans moteur externe à préserver. Les simulations sont bornées et les figures sont du SVG statique.


## 1. Pourquoi apprendre un modèle ?

En model-free, l’agent ajuste directement une valeur ou une politique à partir des transitions vécues. Un modèle appris estime plutôt la conséquence d’une action : $(s,a) \mapsto (r,s')$. Il permet de produire des transitions simulées et de mettre à jour la valeur sans nouvelle interaction avec le monde réel.

| Approche | Apprentissage | Atout | Risque |
|---|---|---|---|
| Model-free | $Q(s,a)$ ou politique | Pas de biais de modèle | Besoin d’interactions |
| Model-based | Valeurs et modèle $\hat P,\hat R$ | Échantillons réels mieux exploités | Un modèle erroné peut propager une mauvaise valeur |

Dyna (Sutton, 1990) entrelace trois opérations : agir, mémoriser la transition, puis rejouer un nombre fixé de transitions du modèle avec la même mise à jour de Q-Learning. La planification consomme du calcul, tandis que le pas réel mesure l’interaction.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

record struct State(int Row, int Col);
record struct Transition(double Reward, int NextState, bool Done);
record PlotSvg(string Content);
string SvgText(string value) => System.Net.WebUtility.HtmlEncode(value);
Formatter.Register<PlotSvg>((value, writer) => writer.Write(value.Content), "text/html");

const int Seed = 42;
const double Gamma = 0.95;
const double Alpha = 0.5;
const double Epsilon = 0.1;
const int MaxStepsPerEpisode = 2000;
var actionNames = new[] { "haut", "bas", "gauche", "droite" };
var moves = new[] { new State(-1, 0), new State(1, 0), new State(0, -1), new State(0, 1) };
Console.WriteLine($"Runtime .NET {Environment.Version}; culture {CultureInfo.CurrentCulture.Name}; graines explicites, aucun paquet NuGet.");


Runtime .NET 10.0.14; culture ; graines explicites, aucun paquet NuGet.


## 2. Labyrinthe de Sutton & Barto

Le Dyna Maze (chapitre 8, figure 8.2) est une grille déterministe 6 × 9. Aller contre un mur ou une bordure laisse l’agent sur place. Seul l’arrivée au but rapporte +1 et termine l’épisode. L’oracle calculé plus bas vérifie la distance optimale au lieu de supposer cette valeur.


In [2]:
class Maze
{
    private static readonly State[] Moves = [new(-1, 0), new(1, 0), new(0, -1), new(0, 1)];
    public int Height { get; }
    public int Width { get; }
    public State Start { get; }
    public State Goal { get; }
    public HashSet<State> Obstacles { get; set; }
    public int StateCount => Height * Width;
    public const int ActionCount = 4;

    public Maze(int height = 6, int width = 9, Nullable<State> start = null, Nullable<State> goal = null,
        IEnumerable<State> obstacles = null)
    {
        Height = height;
        Width = width;
        Start = start ?? new State(2, 0);
        Goal = goal ?? new State(0, 8);
        Obstacles = obstacles?.ToHashSet() ?? new HashSet<State>
        {
            new(1, 2), new(2, 2), new(3, 2), new(4, 5), new(0, 7), new(1, 7), new(2, 7)
        };
        if (Obstacles.Contains(Start) || Obstacles.Contains(Goal))
            throw new ArgumentException("Le départ et le but doivent être libres.");
    }

    public int Index(State state) => state.Row * Width + state.Col;
    public State StateAt(int index) => new(index / Width, index % Width);

    public (State Next, double Reward, bool Done) Step(State state, int action)
    {
        if (action is < 0 or >= ActionCount) throw new ArgumentOutOfRangeException(nameof(action));
        var move = Moves[action];
        var next = new State(state.Row + move.Row, state.Col + move.Col);
        if (next.Row < 0 || next.Row >= Height || next.Col < 0 || next.Col >= Width || Obstacles.Contains(next))
            next = state;
        return next == Goal ? (next, 1.0, true) : (next, 0.0, false);
    }
}

string DrawMaze(Maze maze, string title)
{
    const int cell = 48, margin = 34;
    var svg = new StringBuilder();
    svg.Append($"<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='{SvgText(title)}' viewBox='0 0 {maze.Width * cell + margin * 2} {maze.Height * cell + margin * 2}' width='540'>");
    svg.Append("<rect width='100%' height='100%' fill='white'/>");
    for (var row = 0; row < maze.Height; row++)
    for (var col = 0; col < maze.Width; col++)
    {
        var state = new State(row, col);
        var x = margin + col * cell; var y = margin + row * cell;
        var fill = maze.Obstacles.Contains(state) ? "#252a34" : "#ffffff";
        svg.Append($"<rect x='{x}' y='{y}' width='{cell}' height='{cell}' fill='{fill}' stroke='#9aa3ad' stroke-width='1'/>");
        if (state == maze.Start) svg.Append($"<text x='{x + cell / 2}' y='{y + cell / 2 + 6}' text-anchor='middle' font-family='sans-serif' font-weight='bold' fill='#245ca6'>S</text>");
        if (state == maze.Goal) svg.Append($"<text x='{x + cell / 2}' y='{y + cell / 2 + 6}' text-anchor='middle' font-family='sans-serif' font-weight='bold' fill='#267344'>G</text>");
    }
    svg.Append($"<text x='{margin}' y='20' font-family='sans-serif' font-size='15'>{SvgText(title)}</text></svg>");
    return svg.ToString();
}

string MazeGroup(string svgContent, string transform)
{
    var firstTagEnd = svgContent.IndexOf('>');
    var closingTagStart = svgContent.LastIndexOf("</svg>", StringComparison.Ordinal);
    if (firstTagEnd < 0 || closingTagStart <= firstTagEnd)
        throw new ArgumentException("Le SVG de la carte n’a pas une structure complète.");
    return $"<g transform='{transform}'>{svgContent[(firstTagEnd + 1)..closingTagStart]}</g>";
}

var maze = new Maze();
display(new PlotSvg(DrawMaze(maze, "Dyna Maze — Sutton & Barto, figure 8.2")));
Console.WriteLine($"États tabulaires : {maze.StateCount} ; actions : {Maze.ActionCount} ; murs : {maze.Obstacles.Count} ; départ {maze.Start}, but {maze.Goal}.");


G S Dyna Maze — Sutton & Barto, figure 8.2

États tabulaires : 54 ; actions : 4 ; murs : 7 ; départ State { Row = 2, Col = 0 }, but State { Row = 0, Col = 8 }.


Une seule transition terminale vaut +1 ; les murs et les bords ne terminent pas l’épisode. La carte affichée est la carte utilisée par les expériences suivantes.


In [3]:
int ShortestPathLength(Maze env)
{
    var distance = Enumerable.Repeat(-1, env.StateCount).ToArray();
    var queue = new Queue<State>();
    distance[env.Index(env.Start)] = 0;
    queue.Enqueue(env.Start);
    while (queue.Count > 0)
    {
        var state = queue.Dequeue();
        if (state == env.Goal) return distance[env.Index(state)];
        for (var action = 0; action < Maze.ActionCount; action++)
        {
            var (next, _, _) = env.Step(state, action);
            var index = env.Index(next);
            if (distance[index] >= 0) continue;
            distance[index] = distance[env.Index(state)] + 1;
            queue.Enqueue(next);
        }
    }
    return -1;
}

var optimalSteps = ShortestPathLength(maze);
if (optimalSteps != 14) throw new InvalidOperationException($"Le contrôle BFS attend 14 pas, résultat obtenu : {optimalSteps}.");
foreach (var state in Enumerable.Range(0, maze.StateCount).Select(maze.StateAt).Where(state => !maze.Obstacles.Contains(state)))
for (var action = 0; action < Maze.ActionCount; action++)
{
    var (next, reward, done) = maze.Step(state, action);
    if (done != (next == maze.Goal) || reward != (done ? 1.0 : 0.0))
        throw new InvalidOperationException("Incohérence entre but, terminalité et récompense.");
}
Console.WriteLine($"Oracle BFS indépendant : chemin optimal = {optimalSteps} pas ; terminalité/récompense cohérentes sur toutes les transitions depuis une case libre.");


Oracle BFS indépendant : chemin optimal = 14 pas ; terminalité/récompense cohérentes sur toutes les transitions depuis une case libre.


## 3. Référence model-free : Q-Learning

La mise à jour est $Q(s,a) \leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$. Sur une transition terminale, la cible est simplement $r$ : aucune valeur de l’état suivant n’est bootstrapée. Les dix graines servent à lisser le caractère aléatoire des trajectoires.


In [4]:
int EpsilonGreedy(double[,] q, int state, double epsilon, Random rng)
{
    if (rng.NextDouble() < epsilon) return rng.Next(Maze.ActionCount);
    var max = Enumerable.Range(0, Maze.ActionCount).Max(action => q[state, action]);
    var ties = Enumerable.Range(0, Maze.ActionCount).Where(action => q[state, action] == max).ToArray();
    return ties[rng.Next(ties.Length)];
}

void UpdateQ(double[,] q, int state, int action, Transition transition, double alpha, double gamma)
{
    var target = transition.Reward + (transition.Done ? 0.0 : gamma * Enumerable.Range(0, Maze.ActionCount).Max(a => q[transition.NextState, a]));
    q[state, action] += alpha * (target - q[state, action]);
}

(int[] Steps, double[,] Q) QLearning(Maze env, int episodes = 50, double alpha = Alpha,
    double gamma = Gamma, double epsilon = Epsilon, int seed = 0)
{
    var rng = new Random(seed);
    var q = new double[env.StateCount, Maze.ActionCount];
    var steps = new int[episodes];
    for (var episode = 0; episode < episodes; episode++)
    {
        var state = env.Start;
        var t = 0;
        for (t = 1; t <= MaxStepsPerEpisode; t++)
        {
            var index = env.Index(state);
            var action = EpsilonGreedy(q, index, epsilon, rng);
            var (next, reward, done) = env.Step(state, action);
            UpdateQ(q, index, action, new Transition(reward, env.Index(next), done), alpha, gamma);
            state = next;
            if (done) break;
        }
        steps[episode] = Math.Min(t, MaxStepsPerEpisode);
    }
    return (steps, q);
}

string DrawLineChart(string title, string xLabel, string yLabel, IReadOnlyList<(string Label, double[] Values, string Color)> curves,
    double? reference = null, int width = 820, int height = 440)
{
    const int left = 72, top = 48, right = 190, bottom = 62;
    var plotWidth = width - left - right; var plotHeight = height - top - bottom;
    var allValues = curves.SelectMany(curve => curve.Values).ToList();
    if (reference.HasValue) allValues.Add(reference.Value);
    var min = Math.Min(0, allValues.Min()); var max = allValues.Max();
    if (max <= min) max = min + 1;
    var svg = new StringBuilder();
    svg.Append($"<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='{SvgText(title)}' viewBox='0 0 {width} {height}' width='{width}'>");
    svg.Append("<rect width='100%' height='100%' fill='white'/>");
    svg.Append($"<text x='{width / 2}' y='25' text-anchor='middle' font-family='sans-serif' font-size='17'>{SvgText(title)}</text>");
    for (var tick = 0; tick <= 4; tick++)
    {
        var y = top + tick * plotHeight / 4.0;
        var value = max - tick * (max - min) / 4.0;
        svg.Append($"<line x1='{left}' y1='{y:F1}' x2='{left + plotWidth}' y2='{y:F1}' stroke='#e1e5e9'/>");
        svg.Append($"<text x='{left - 8}' y='{y + 4:F1}' text-anchor='end' font-family='sans-serif' font-size='11' fill='#414a52'>{value:F1}</text>");
    }
    svg.Append($"<line x1='{left}' y1='{top + plotHeight}' x2='{left + plotWidth}' y2='{top + plotHeight}' stroke='#414a52'/><line x1='{left}' y1='{top}' x2='{left}' y2='{top + plotHeight}' stroke='#414a52'/>");
    if (reference.HasValue)
    {
        var y = top + (max - reference.Value) / (max - min) * plotHeight;
        svg.Append($"<line x1='{left}' y1='{y:F1}' x2='{left + plotWidth}' y2='{y:F1}' stroke='#555' stroke-dasharray='6 5'/>");
        svg.Append($"<text x='{left + plotWidth + 10}' y='{y + 4:F1}' font-family='sans-serif' font-size='11' fill='#414a52'>optimum BFS</text>");
    }
    foreach (var curve in curves)
    {
        var points = curve.Values.Select((value, index) => $"{left + (curve.Values.Length == 1 ? 0 : index * plotWidth / (double)(curve.Values.Length - 1)):F1},{top + (max - value) / (max - min) * plotHeight:F1}");
        svg.Append($"<polyline fill='none' stroke='{curve.Color}' stroke-width='2' points='{string.Join(" ", points)}'/>");
    }
    for (var tick = 0; tick <= 4; tick++)
    {
        var x = left + tick * plotWidth / 4.0;
        var sample = 1 + (curves[0].Values.Length - 1) * tick / 4.0;
        svg.Append($"<line x1='{x:F1}' y1='{top + plotHeight}' x2='{x:F1}' y2='{top + plotHeight + 4}' stroke='#414a52'/>");
        svg.Append($"<text x='{x:F1}' y='{top + plotHeight + 18}' text-anchor='middle' font-family='sans-serif' font-size='10' fill='#414a52'>{sample:F0}</text>");
    }
    svg.Append($"<text x='{left + plotWidth / 2}' y='{height - 15}' text-anchor='middle' font-family='sans-serif' font-size='12'>{SvgText(xLabel)}</text>");
    svg.Append($"<text transform='translate(18 {top + plotHeight / 2}) rotate(-90)' text-anchor='middle' font-family='sans-serif' font-size='12'>{SvgText(yLabel)}</text>");
    for (var i = 0; i < curves.Count; i++)
    {
        var y = top + 15 + i * 22;
        svg.Append($"<line x1='{left + plotWidth + 10}' y1='{y}' x2='{left + plotWidth + 30}' y2='{y}' stroke='{curves[i].Color}' stroke-width='2'/><text x='{left + plotWidth + 36}' y='{y + 4}' font-family='sans-serif' font-size='11'>{SvgText(curves[i].Label)}</text>");
    }
    svg.Append("</svg>");
    return svg.ToString();
}

const int Runs = 10, Episodes = 50;
var qRuns = Enumerable.Range(0, Runs).Select(seed => QLearning(maze, Episodes, seed: seed).Steps).ToArray();
var qMean = Enumerable.Range(0, Episodes).Select(episode => qRuns.Average(run => run[episode])).ToArray();
display(new PlotSvg(DrawLineChart("Référence model-free sur Dyna Maze", "Épisode", "Pas par épisode (moyenne sur 10 graines)",
    [("Q-Learning", qMean, "#2368a2")], optimalSteps)));
Console.WriteLine($"Q-Learning, pas moyens : épisode 1 = {qMean[0]:F1}, épisode 10 = {qMean[9]:F1}, épisode 50 = {qMean[^1]:F1}.");


Référence model-free sur Dyna Maze 858.7 644.0 429.4 214.7 0.0 optimum BFS <polyline fill='none' stroke='#2368a2' stroke-width='2' points='72.0,48.0 83.4,148.0 94.8,198.7 106.2,280.1 117.6,272.9 128.9,266.8 140.3,299.1 151.7,322.4 163.1,345.1 174.5,350.0 185.9,355.1 197.3,338.7 208.7,346.2 220.0,349.6 231.4,346.2 242.8,354.1 254.2,341.9 265.6,356.2 277.0,362.3 288.4,365.8 299.8,361.1 311.1,366.7 322.5,365.5 333.9,369.2 345.3,369.9 356.7,369.2 368.1,370.6 379.5,371.1 390.9,370.3 402.2,370.8 413.6,370.2 425.0,371.7 436.4,371.4 447.8,370.9 459.2,371.3 470.6,370.3 482.0,369.7 493.3,371.0 504.7,371.3 516.1,370.8 527.5,370.7 538.9,370.8 550.3,371.0 561.7,369.1 573.1,370.5 584.4,370.9 595.8,371.6 607.2,371.3 618.6,370.1 630.0,371.4'/> 1 13 26 38 50 Épisode Pas par épisode (moyenne sur 10 graines) Q-Learning

Q-Learning, pas moyens : épisode 1 = 858.7, épisode 10 = 72.8, épisode 50 = 17.3.


La courbe et les trois repères numériques proviennent des trajectoires réellement exécutées. La ligne BFS est une borne structurelle du labyrinthe. Une convergence lente est plausible avec une récompense éparse, mais son ampleur dépend des graines et du départage aléatoire.


## 4. Un modèle tabulaire déterministe

Pour chaque paire $(s,a)$ observée, le modèle mémorise $(r,s',done)$. Une paire répétée remplace sa valeur ; l’échantillonnage ne porte que sur des clés réellement enregistrées. En environnement stochastique, ce modèle à résultat unique ne suffirait pas : il faudrait estimer une distribution.


In [5]:
sealed class DeterministicModel
{
    private readonly Dictionary<(int State, int Action), Transition> transitions = new();
    private readonly List<(int State, int Action)> keys = new();
    public IReadOnlyDictionary<(int State, int Action), Transition> Transitions => transitions;
    public int Count => transitions.Count;

    public void Update(int state, int action, double reward, int nextState, bool done)
    {
        var key = (state, action);
        if (!transitions.ContainsKey(key)) keys.Add(key);
        transitions[key] = new Transition(reward, nextState, done);
    }

    public ((int State, int Action) Pair, Transition Value) Sample(Random rng)
    {
        if (keys.Count == 0) throw new InvalidOperationException("Le modèle vide ne peut pas être échantillonné.");
        var pair = keys[rng.Next(keys.Count)];
        return (pair, transitions[pair]);
    }
}

var demoRandom = new Random(Seed);
var demoModel = new DeterministicModel();
var demoState = maze.Start;
for (var step = 0; step < 200; step++)
{
    var action = demoRandom.Next(Maze.ActionCount);
    var (next, reward, done) = maze.Step(demoState, action);
    demoModel.Update(maze.Index(demoState), action, reward, maze.Index(next), done);
    demoState = done ? maze.Start : next;
}

var possiblePairs = maze.StateCount * Maze.ActionCount;
var sampled = demoModel.Sample(demoRandom);
if (!demoModel.Transitions.ContainsKey(sampled.Pair)) throw new InvalidOperationException("Le modèle a échantillonné une paire inconnue.");
if (demoModel.Transitions.Any(entry => entry.Value.Done != (entry.Value.Reward == 1.0))) throw new InvalidOperationException("Récompense et terminalité du modèle incohérentes.");
Console.WriteLine($"Après 200 pas aléatoires : {demoModel.Count}/{possiblePairs} paires observées ({100.0 * demoModel.Count / possiblePairs:F0} %).");
Console.WriteLine($"Transition simulée : état {sampled.Pair.State}, action {sampled.Pair.Action} ({actionNames[sampled.Pair.Action]}) → état {sampled.Value.NextState}, récompense {sampled.Value.Reward:F0}, terminal={sampled.Value.Done}.");


Après 200 pas aléatoires : 71/216 paires observées (33 %).


Transition simulée : état 6, action 1 (bas) → état 15, récompense 0, terminal=False.


Le dénominateur comprend les 54 cases de la grille, murs compris, multipliées par quatre actions ; c’est l’espace tabulaire déclaré, pas le nombre d’états physiquement accessibles. L’échantillon est tiré d’une paire connue et restitue la récompense et la terminalité enregistrées.


## 5. Dyna-Q : agir, apprendre, planifier

Après la mise à jour réelle, l’agent mémorise la transition, puis tire $n$ paires observées au hasard dans le modèle. Chaque tirage applique la même règle de Q-Learning. Avec $n=0$, le code retombe sur la baseline model-free. La comparaison conserve dix graines et cinquante épisodes pour chaque budget de planification.


In [6]:
(int[] Steps, double[,] Q, DeterministicModel Model) DynaQ(Maze env, int episodes = 50,
    int planningSteps = 5, double alpha = Alpha, double gamma = Gamma,
    double epsilon = Epsilon, int seed = 0)
{
    var rng = new Random(seed);
    var q = new double[env.StateCount, Maze.ActionCount];
    var model = new DeterministicModel();
    var steps = new int[episodes];
    for (var episode = 0; episode < episodes; episode++)
    {
        var state = env.Start;
        var t = 0;
        for (t = 1; t <= MaxStepsPerEpisode; t++)
        {
            var stateIndex = env.Index(state);
            var action = EpsilonGreedy(q, stateIndex, epsilon, rng);
            var (next, reward, done) = env.Step(state, action);
            var transition = new Transition(reward, env.Index(next), done);
            UpdateQ(q, stateIndex, action, transition, alpha, gamma);
            model.Update(stateIndex, action, reward, transition.NextState, done);
            for (var plan = 0; plan < planningSteps; plan++)
            {
                var (pair, simulated) = model.Sample(rng);
                UpdateQ(q, pair.State, pair.Action, simulated, alpha, gamma);
            }
            state = next;
            if (done) break;
        }
        steps[episode] = Math.Min(t, MaxStepsPerEpisode);
    }
    return (steps, q, model);
}

if (!qRuns[0].SequenceEqual(DynaQ(maze, Episodes, planningSteps: 0, seed: 0).Steps))
    throw new InvalidOperationException("Avec n=0, Dyna-Q doit reproduire la baseline pour la même graine.");
Console.WriteLine("Contrôle n=0 : Dyna-Q reproduit exactement les pas de Q-Learning sur la graine 0.");
var planningLevels = new[] { 0, 5, 50 };
var planningResults = new Dictionary<int, double[]>();
foreach (var planning in planningLevels)
{
    var runsForLevel = Enumerable.Range(0, Runs).Select(seed => DynaQ(maze, Episodes, planning, seed: seed).Steps).ToArray();
    planningResults[planning] = Enumerable.Range(0, Episodes).Select(episode => runsForLevel.Average(run => run[episode])).ToArray();
}
var planningCurves = planningLevels.Select((planning, index) =>
    (planning == 0 ? "n=0 (Q-Learning)" : $"n={planning}", planningResults[planning], new[] { "#2368a2", "#df7b24", "#2b8b57" }[index])).ToArray();
display(new PlotSvg(DrawLineChart("Dyna-Q : budget de planification", "Épisode", "Pas par épisode (moyenne sur 10 graines)", planningCurves, optimalSteps)));
foreach (var planning in planningLevels)
{
    var curve = planningResults[planning];
    var firstUnder = Array.FindIndex(curve, value => value < 25) + 1;
    Console.WriteLine($"n={planning,2} : premier épisode sous 25 pas = {(firstUnder == 0 ? "non atteint" : firstUnder)} ; épisodes 1/10/50 = {curve[0]:F1}/{curve[9]:F1}/{curve[^1]:F1}.");
}
Console.WriteLine("Chaque courbe compte 10 graines × 50 épisodes ; n compte des mises à jour simulées, pas des interactions réelles.");


Contrôle n=0 : Dyna-Q reproduit exactement les pas de Q-Learning sur la graine 0.


Dyna-Q : budget de planification 1069.3 802.0 534.6 267.3 0.0 optimum BFS <polyline fill='none' stroke='#2368a2' stroke-width='2' points='72.0,113.0 83.4,193.3 94.8,234.0 106.2,299.4 117.6,293.6 128.9,288.7 140.3,314.6 151.7,333.4 163.1,351.6 174.5,355.5 185.9,359.6 197.3,346.4 208.7,352.5 220.0,355.2 231.4,352.4 242.8,358.8 254.2,349.0 265.6,360.5 277.0,365.4 288.4,368.2 299.8,364.5 311.1,369.0 322.5,368.0 333.9,370.9 345.3,371.5 356.7,370.9 368.1,372.0 379.5,372.4 390.9,371.8 402.2,372.2 413.6,371.7 425.0,372.9 436.4,372.7 447.8,372.3 459.2,372.6 470.6,371.8 482.0,371.3 493.3,372.4 504.7,372.6 516.1,372.2 527.5,372.1 538.9,372.2 550.3,372.4 561.7,370.9 573.1,372.0 584.4,372.3 595.8,372.8 607.2,372.6 618.6,371.6 630.0,372.7'/><polyline fill='none' stroke='#df7b24' stroke-width='2' points='72.0,48.0 83.4,329.2 94.8,357.1 106.2,368.6 117.6,371.2 128.9,371.3 140.3,372.4 151.7,372.8 163.1,372.3 174.5,372.8 185.9,372.4 197.3,372.8 208.7,373.1 220.0,372.4 231.4,372.7 242.8,372.7 254.2,372.7 265.6,372.7 277.0,372.9 288.4,373.0 299.8,372.4 311.1,372.8 322.5,372.8 333.9,372.7 345.3,373.0 356.7,372.7 368.1,372.7 379.5,373.0 390.9,372.9 402.2,373.1 413.6,372.9 425.0,372.9 436.4,373.0 447.8,372.5 459.2,373.0 470.6,373.0 482.0,372.9 493.3,372.6 504.7,372.8 516.1,372.8 527.5,373.3 538.9,373.3 550.3,372.8 561.7,373.0 573.1,372.8 584.4,372.8 595.8,372.9 607.2,373.0 618.6,373.1 630.0,373.0'/><polyline fill='none' stroke='#2b8b57' stroke-width='2' points='72.0,147.8 83.4,366.1 94.8,372.7 106.2,373.0 117.6,372.9 128.9,373.0 140.3,373.2 151.7,372.8 163.1,372.9 174.5,372.9 185.9,372.9 197.3,372.8 208.7,372.9 220.0,373.1 231.4,372.8 242.8,373.0 254.2,373.2 265.6,372.7 277.0,372.6 288.4,372.8 299.8,372.8 311.1,373.0 322.5,373.0 333.9,373.0 345.3,372.9 356.7,373.1 368.1,373.0 379.5,372.7 390.9,373.3 402.2,372.8 413.6,373.1 425.0,372.8 436.4,372.8 447.8,373.1 459.2,372.8 470.6,372.7 482.0,372.9 493.3,372.8 504.7,373.1 516.1,372.8 527.5,373.2 538.9,372.8 550.3,372.6 561.7,372.7 573.1,373.1 584.4,373.1 595.8,373.1 607.2,372.8 618.6,373.3 630.0,373.0'/> 1 13 26 38 50 Épisode Pas par épisode (moyenne sur 10 graines) n=0 (Q-Learning) n=5 n=50

n= 0 : premier épisode sous 25 pas = 24 ; épisodes 1/10/50 = 858.7/72.8/17.3.


n= 5 : premier épisode sous 25 pas = 5 ; épisodes 1/10/50 = 1069.3/16.9/16.2.


n=50 : premier épisode sous 25 pas = 3 ; épisodes 1/10/50 = 745.8/16.4/16.1.


Chaque courbe compte 10 graines × 50 épisodes ; n compte des mises à jour simulées, pas des interactions réelles.


Les nombres ci-dessus donnent le résultat de ce run C# : on compare le premier franchissement du seuil de 25 pas et la fin des courbes, sans imposer à l’avance que plus de planification gagne toujours. Même si une courbe descend plus vite, elle consomme davantage de calcul pour un nombre donné de pas réels.


## 6. Modèle périmé : Blocking Maze et Dyna-Q+

Le Blocking Maze change au pas réel 1 000 : le passage par la droite est fermé et celui de gauche ouvert. Dyna-Q+ ajoute au **ciblage planifié** le bonus $\kappa\sqrt{\tau(s,a)}$, où $\tau$ mesure le temps écoulé depuis la dernière vraie visite. Depuis un état visité, les actions encore inconnues deviennent des boucles sur place planifiables ; elles peuvent ainsi attirer l’exploration. Les récompenses du graphique restent celles des interactions réelles.


In [7]:
sealed class ChangingMaze : Maze
{
    public HashSet<State> ObstaclesBefore { get; }
    public HashSet<State> ObstaclesAfter { get; }
    public int SwitchTime { get; }

    public ChangingMaze(IEnumerable<State> before, IEnumerable<State> after, int switchTime,
        Nullable<State> start = null, Nullable<State> goal = null)
        : base(start: start ?? new State(5, 3), goal: goal ?? new State(0, 8), obstacles: before)
    {
        ObstaclesBefore = before.ToHashSet();
        ObstaclesAfter = after.ToHashSet();
        SwitchTime = switchTime;
    }

    public (State Next, double Reward, bool Done) Step(State state, int action, int globalStep)
    {
        Obstacles = globalStep < SwitchTime ? ObstaclesBefore : ObstaclesAfter;
        return base.Step(state, action);
    }
}

var wallBefore = Enumerable.Range(0, 8).Select(col => new State(3, col)).ToHashSet();
var wallAfter = Enumerable.Range(1, 8).Select(col => new State(3, col)).ToHashSet();
const int SwitchTime = 1000, TotalSteps = 3000;
var blockingMaze = new ChangingMaze(wallBefore, wallAfter, SwitchTime);
display(new PlotSvg($"<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Blocking Maze avant et après le changement' viewBox='0 0 1120 380' width='900'><rect width='100%' height='100%' fill='white'/>{MazeGroup(DrawMaze(new Maze(start: blockingMaze.Start, goal: blockingMaze.Goal, obstacles: wallBefore), "Avant : ouverture à droite"), "translate(0 0)")}{MazeGroup(DrawMaze(new Maze(start: blockingMaze.Start, goal: blockingMaze.Goal, obstacles: wallAfter), "Après : ouverture à gauche"), "translate(560 0)")}</svg>"));
Console.WriteLine($"Changement de carte au pas {SwitchTime} sur un budget de {TotalSteps} interactions réelles ; murs avant/après : {wallBefore.Count}/{wallAfter.Count}.");


G S Avant : ouverture à droite G S Après : ouverture à gauche

Changement de carte au pas 1000 sur un budget de 3000 interactions réelles ; murs avant/après : 8/8.


Les deux cartes ont le même départ, le même but et huit cases de mur ; seul le côté de l’ouverture change. L’agent ne reçoit aucune notification du changement : il le découvre par ses transitions réelles.


In [8]:
double[] DynaQSteps(ChangingMaze env, int totalSteps, int planningSteps = 10,
    double kappa = 0.0, double alpha = Alpha, double gamma = Gamma,
    double epsilon = Epsilon, int seed = 0)
{
    var rng = new Random(seed);
    var q = new double[env.StateCount, Maze.ActionCount];
    var model = new DeterministicModel();
    var age = new double[env.StateCount, Maze.ActionCount];
    var cumulative = new double[totalSteps];
    var rewardTotal = 0.0;
    var state = env.Start;
    for (var t = 0; t < totalSteps; t++)
    {
        var stateIndex = env.Index(state);
        var action = EpsilonGreedy(q, stateIndex, epsilon, rng);
        var (next, reward, done) = env.Step(state, action, t);
        var nextIndex = env.Index(next);
        UpdateQ(q, stateIndex, action, new Transition(reward, nextIndex, done), alpha, gamma);
        for (var s = 0; s < env.StateCount; s++)
        for (var a = 0; a < Maze.ActionCount; a++) age[s, a] += 1.0;
        age[stateIndex, action] = 0.0;
        model.Update(stateIndex, action, reward, nextIndex, done);
        if (kappa > 0)
        {
            for (var candidate = 0; candidate < Maze.ActionCount; candidate++)
                if (!model.Transitions.ContainsKey((stateIndex, candidate)))
                    model.Update(stateIndex, candidate, 0.0, stateIndex, false);
        }
        for (var plan = 0; plan < planningSteps; plan++)
        {
            var (pair, simulated) = model.Sample(rng);
            var bonus = kappa > 0 ? kappa * Math.Sqrt(age[pair.State, pair.Action]) : 0.0;
            var target = simulated.Reward + bonus + (simulated.Done ? 0.0 : gamma * Enumerable.Range(0, Maze.ActionCount).Max(a => q[simulated.NextState, a]));
            q[pair.State, pair.Action] += alpha * (target - q[pair.State, pair.Action]);
        }
        rewardTotal += reward;
        cumulative[t] = rewardTotal;
        state = done ? env.Start : next;
    }
    return cumulative;
}

double[] RunBlocking(double kappa, int seeds = 5) => Enumerable.Range(0, seeds)
    .Select(seed => DynaQSteps(new ChangingMaze(wallBefore, wallAfter, SwitchTime), TotalSteps, 10, kappa, seed: seed))
    .Aggregate((sum, curve) => sum.Zip(curve, (left, right) => left + right).ToArray())
    .Select(value => value / seeds).ToArray();

var cumulativeDyna = RunBlocking(0.0);
var cumulativeDynaPlus = RunBlocking(1e-3);
var blockingChart = DrawLineChart("Blocking Maze : récompenses réellement obtenues", "Pas réel", "Récompense cumulée (moyenne sur 5 graines)",
    [("Dyna-Q, κ=0", cumulativeDyna, "#2368a2"), ("Dyna-Q+, κ=0,001", cumulativeDynaPlus, "#df7b24")], width: 900, height: 460);
var switchX = 72 + (SwitchTime - 1) * (900 - 72 - 190) / (double)(TotalSteps - 1);
blockingChart = blockingChart.Replace("</svg>", $"<line x1='{switchX:F1}' y1='48' x2='{switchX:F1}' y2='398' stroke='#b52b2b' stroke-dasharray='5 4'/><text x='{switchX + 6:F1}' y='65' font-family='sans-serif' font-size='11' fill='#9e2525'>changement à {SwitchTime}</text></svg>");
display(new PlotSvg(blockingChart));
Console.WriteLine($"Récompense cumulée finale (5 graines) : Dyna-Q = {cumulativeDyna[^1]:F1} ; Dyna-Q+ = {cumulativeDynaPlus[^1]:F1}.");
Console.WriteLine($"À t=1000 : {cumulativeDyna[SwitchTime - 1]:F1} / {cumulativeDynaPlus[SwitchTime - 1]:F1} ; à t=3000 : {cumulativeDyna[^1]:F1} / {cumulativeDynaPlus[^1]:F1}.");


Blocking Maze : récompenses réellement obtenues 113.0 84.8 56.5 28.2 0.0 <polyline fill='none' stroke='#2368a2' stroke-width='2' points='72.0,398.0 72.2,398.0 72.4,398.0 72.6,398.0 72.9,398.0 73.1,398.0 73.3,398.0 73.5,398.0 73.7,398.0 73.9,398.0 74.1,398.0 74.3,398.0 74.6,398.0 74.8,398.0 75.0,398.0 75.2,398.0 75.4,398.0 75.6,398.0 75.8,398.0 76.0,398.0 76.3,398.0 76.5,398.0 76.7,398.0 76.9,398.0 77.1,398.0 77.3,398.0 77.5,398.0 77.7,398.0 78.0,398.0 78.2,398.0 78.4,398.0 78.6,398.0 78.8,398.0 79.0,398.0 79.2,398.0 79.4,398.0 79.7,398.0 79.9,398.0 80.1,398.0 80.3,398.0 80.5,398.0 80.7,398.0 80.9,398.0 81.1,398.0 81.4,398.0 81.6,398.0 81.8,398.0 82.0,398.0 82.2,398.0 82.4,398.0 82.6,398.0 82.8,398.0 83.1,398.0 83.3,398.0 83.5,398.0 83.7,398.0 83.9,398.0 84.1,398.0 84.3,398.0 84.6,398.0 84.8,398.0 85.0,398.0 85.2,398.0 85.4,398.0 85.6,398.0 85.8,398.0 86.0,398.0 86.3,398.0 86.5,398.0 86.7,398.0 86.9,398.0 87.1,398.0 87.3,398.0 87.5,398.0 87.7,398.0 88.0,398.0 88.2,398.0 88.4,398.0 88.6,398.0 88.8,398.0 89.0,398.0 89.2,398.0 89.4,398.0 89.7,398.0 89.9,398.0 90.1,398.0 90.3,398.0 90.5,398.0 90.7,398.0 90.9,398.0 91.1,398.0 91.4,398.0 91.6,398.0 91.8,398.0 92.0,398.0 92.2,398.0 92.4,398.0 92.6,398.0 92.8,398.0 93.1,398.0 93.3,398.0 93.5,398.0 93.7,398.0 93.9,398.0 94.1,398.0 94.3,398.0 94.6,398.0 94.8,398.0 95.0,398.0 95.2,398.0 95.4,398.0 95.6,398.0 95.8,398.0 96.0,398.0 96.3,398.0 96.5,398.0 96.7,398.0 96.9,398.0 97.1,398.0 97.3,398.0 97.5,398.0 97.7,398.0 98.0,398.0 98.2,398.0 98.4,398.0 98.6,398.0 98.8,398.0 99.0,398.0 99.2,398.0 99.4,398.0 99.7,398.0 99.9,398.0 100.1,398.0 100.3,398.0 100.5,398.0 100.7,398.0 100.9,398.0 101.1,398.0 101.4,398.0 101.6,398.0 101.8,398.0 102.0,398.0 102.2,398.0 102.4,398.0 102.6,398.0 102.8,398.0 103.1,398.0 103.3,398.0 103.5,398.0 103.7,398.0 103.9,398.0 104.1,398.0 104.3,398.0 104.5,398.0 104.8,398.0 105.0,398.0 105.2,398.0 105.4,398.0 105.6,398.0 105.8,398.0 106.0,398.0 106.3,398.0 106.5,398.0 106.7,398.0 106.9,398.0 107.1,398.0 107.3,398.0 107.5,398.0 107.7,398.0 108.0,398.0 108.2,398.0 108.4,398.0 108.6,398.0 108.8,398.0 109.0,398.0 109.2,398.0 109.4,398.0 109.7,398.0 109.9,398.0 110.1,398.0 110.3,398.0 110.5,398.0 110.7,398.0 110.9,398.0 111.1,398.0 111.4,398.0 111.6,398.0 111.8,398.0 112.0,398.0 112.2,398.0 112.4,398.0 112.6,398.0 112.8,398.0 113.1,398.0 113.3,398.0 113.5,398.0 113.7,398.0 113.9,398.0 114.1,398.0 114.3,398.0 114.5,398.0 114.8,398.0 115.0,398.0 115.2,398.0 115.4,398.0 115.6,398.0 115.8,398.0 116.0,398.0 116.2,398.0 116.5,398.0 116.7,398.0 116.9,398.0 117.1,397.4 117.3,397.4 117.5,397.4 117.7,397.4 118.0,397.4 118.2,397.4 118.4,397.4 118.6,397.4 118.8,397.4 119.0,397.4 119.2,397.4 119.4,397.4 119.7,397.4 119.9,397.4 120.1,397.4 120.3,397.4 120.5,396.8 120.7,396.8 120.9,396.8 121.1,396.8 121.4,396.8 121.6,396.8 121.8,396.8 122.0,396.8 122.2,396.8 122.4,396.8 122.6,396.8 122.8,396.8 123.1,396.8 123.3,396.8 123.5,396.8 123.7,396.8 123.9,396.8 124.1,396.8 124.3,396.8 124.5,396.8 124.8,396.8 125.0,396.8 125.2,396.8 125.4,396.8 125.6,396.8 125.8,396.8 126.0,396.8 126.2,396.8 126.5,396.8 126.7,396.8 126.9,396.8 127.1,396.8 127.3,396.8 127.5,396.8 127.7,396.8 127.9,396.8 128.2,396.8 128.4,396.8 128.6,396.8 128.8,396.8 129.0,396.8 129.2,396.8 129.4,396.8 129.7,396.8 129.9,396.8 130.1,396.8 130.3,396.8 130.5,396.8 130.7,396.8 130.9,396.8 131.1,396.8 131.4,396.8 131.6,396.8 131.8,396.8 132.0,396.1 132.2,396.1 132.4,396.1 132.6,396.1 132.8,396.1 133.1,396.1 133.3,396.1 133.5,396.1 133.7,396.1 133.9,396.1 134.1,396.1 134.3,396.1 134.5,395.5 134.8,395.5 135.0,395.5 135.2,395.5 135.4,395.5 135.6,395.5 135.8,395.5 136.0,395.5 136.2,395.5 136.5,395.5 136.7,395.5 136.9,395.5 137.1,395.5 137.3,395.5 137.5,395.5 137.7,394.9 137.9,394.9 138.2,394.9 138.4,394.9 138.6,394.9 138.8,394.9 139.0,394.9 139.2,394.9 139.4,394.9 139.7,394.9 139.9,394.3 140.1,394.3 140.3,394.3 140.5,394.3 140.7,394.3 140.9,394.3 141.1,393.7 141.4,393.7 141.6,393.7 141.8,393.7 142.0,393.7 142.2,393.7 142.4,393.7 142.6,393.7

Récompense cumulée finale (5 graines) : Dyna-Q = 86.4 ; Dyna-Q+ = 113.0.


À t=1000 : 40.8 / 53.4 ; à t=3000 : 86.4 / 113.0.


Le changement vertical du graphique est tracé au pas 1 000 ; les valeurs imprimées permettent de comparer avant et après, sur cinq graines. La conclusion porte sur ces courbes observées dans cette exécution. Le bonus peut accélérer la redécouverte, mais aussi faire perdre du temps à revérifier des actions inchangées ; aucune supériorité universelle n’est supposée.


## Exercices d’application

Les trois exercices sont des cellules compilables mais non résolues. Complétez-les après les exemples guidés ; chaque cellule garde un point de départ exécutable sans produire une réponse inventée.


### Exercice 1 — Shortcut Maze : découvrir un raccourci

Ouvrez au pas 3 000 un passage à droite dans un mur horizontal, sans fermer l’ancien chemin. Construisez les murs avant/après, lancez 6 000 pas sur cinq graines avec $\kappa=0$ puis $10^{-3}$, et tracez les récompenses cumulées. Comparez numériquement les pentes avant/après le changement.

**Indice** : ligne 3 ; avant, colonnes 1–8 sont des murs ; après, colonnes 1–7. Une pente de segment se calcule par différence des récompenses cumulées divisée par le nombre de pas.


In [9]:
// Exercice 1 — Shortcut Maze
// TODO étudiant : construire les deux ensembles de murs, exécuter 5 graines,
// tracer les deux séries SVG et calculer leurs pentes après le pas 3000.
var shortcutWallBefore = new HashSet<State>();
var shortcutWallAfter = new HashSet<State>();
ChangingMaze shortcutMaze = null!;
if (shortcutMaze is null)
{
    Console.WriteLine("Exercice 1 à compléter : aucune carte ni courbe n’est mesurée en l’état.");
}
else
{
    if (shortcutMaze.SwitchTime != 3000 || !shortcutMaze.ObstaclesBefore.SetEquals(shortcutWallBefore)
        || !shortcutMaze.ObstaclesAfter.SetEquals(shortcutWallAfter))
        throw new InvalidOperationException("La carte ou le pas de changement ne correspond pas à l’énoncé.");
    var beforeLength = ShortestPathLength(new Maze(start: shortcutMaze.Start, goal: shortcutMaze.Goal, obstacles: shortcutWallBefore));
    var afterLength = ShortestPathLength(new Maze(start: shortcutMaze.Start, goal: shortcutMaze.Goal, obstacles: shortcutWallAfter));
    if (afterLength >= beforeLength) throw new InvalidOperationException("Le passage ouvert doit raccourcir le chemin oracle.");
    Console.WriteLine($"Contrôle Shortcut : chemin avant={beforeLength}, après={afterLength} ; vérifier ensuite les pentes empiriques.");
}


Exercice 1 à compléter : aucune carte ni courbe n’est mesurée en l’état.


### Exercice 2 — Prioritized Sweeping

Implémentez une file de priorité sur l’erreur de Bellman absolue $|\delta|$, puis propagez la priorité aux prédécesseurs connus de l’état mis à jour. Comparez au Dyna-Q ($n=5$) sur dix graines et cinquante épisodes.

**Indices** : `PriorityQueue<(int State, int Action), double>` est disponible dans le BCL ; une priorité plus élevée se représente par son opposé. Ignorez les transitions terminales dans le terme bootstrap.


In [10]:
// Exercice 2 — Prioritized Sweeping
// TODO étudiant : renvoyer 50 pas moyens (10 graines) et une Q-table
// issue de la file de priorité et de la propagation aux prédécesseurs.
(double[] Steps, double[,] Q)? PrioritizedSweeping(Maze env, int episodes = 50,
    int planningSteps = 5, double theta = 1e-4, int seed = 0)
{
    Console.WriteLine("Prioritized Sweeping à compléter : aucune série de performance n’est calculée.");
    return null;
}
var prioritizedResult = PrioritizedSweeping(maze, episodes: Episodes);
if (prioritizedResult is null)
{
    Console.WriteLine("Exercice 2 à compléter ; conservez la mise à jour terminale sans bootstrap.");
}
else
{
    var (steps, qPrioritized) = prioritizedResult.Value;
    if (steps.Length != Episodes || qPrioritized.GetLength(0) != maze.StateCount
        || qPrioritized.GetLength(1) != Maze.ActionCount
        || steps.Any(value => !double.IsFinite(value) || value < 1 || value > MaxStepsPerEpisode))
        throw new InvalidOperationException("La série doit contenir 50 longueurs d’épisode valides et une Q-table de bonne forme.");
    var firstBelow = Array.FindIndex(steps, value => value < 25) + 1;
    Console.WriteLine($"Contrôle Prioritized Sweeping : {steps.Length} épisodes, Q={qPrioritized.GetLength(0)}×{qPrioritized.GetLength(1)}, premier seuil <25 = {(firstBelow == 0 ? "non atteint" : firstBelow)}.");
}


Prioritized Sweeping à compléter : aucune série de performance n’est calculée.


Exercice 2 à compléter ; conservez la mise à jour terminale sans bootstrap.


## 7. Planification au moment de la décision : rollouts

Dyna met à jour une table en arrière-plan. Un rollout évalue plutôt chaque action candidate en déroulant des trajectoires futures à partir de l’état courant, ici avec une politique uniforme et le modèle parfait de la carte. MCTS concentre ensuite les simulations sur les branches prometteuses (sélection, expansion, simulation, rétropropagation). AlphaZero utilise les règles du jeu comme modèle ; MuZero apprend un modèle latent.


In [11]:
double[] RolloutActionValues(Maze env, State start, int rollouts = 300, int depth = 60,
    double gamma = Gamma, int seed = 0)
{
    var rng = new Random(seed);
    var estimates = new double[Maze.ActionCount];
    for (var action = 0; action < Maze.ActionCount; action++)
    {
        var total = 0.0;
        for (var rollout = 0; rollout < rollouts; rollout++)
        {
            var state = start;
            var discount = 1.0;
            var returns = 0.0;
            var (next, reward, done) = env.Step(state, action);
            returns += discount * reward;
            state = next;
            for (var step = 0; !done && step < depth; step++)
            {
                discount *= gamma;
                (state, reward, done) = env.Step(state, rng.Next(Maze.ActionCount));
                returns += discount * reward;
            }
            total += returns;
        }
        estimates[action] = total / rollouts;
    }
    return estimates;
}

var probeState = new State(3, 8);
var rolloutValues = RolloutActionValues(maze, probeState, seed: Seed);
foreach (var action in Enumerable.Range(0, Maze.ActionCount).OrderByDescending(action => rolloutValues[action]))
    Console.WriteLine($"{actionNames[action],-8} : {rolloutValues[action]:F3}");
var bestRolloutAction = Array.IndexOf(rolloutValues, rolloutValues.Max());
Console.WriteLine($"Meilleure action mesurée depuis {probeState} : {actionNames[bestRolloutAction]} ; calculée sur 300 rollouts/action, profondeur 60.");


haut     : 0.336


droite   : 0.177


gauche   : 0.094


bas      : 0.083


Meilleure action mesurée depuis State { Row = 3, Col = 8 } : haut ; calculée sur 300 rollouts/action, profondeur 60.


Les scores mesurent une moyenne Monte-Carlo bruitée, pas une valeur optimale. La meilleure action est déterminée par les quatre estimations réellement calculées. Avec une récompense éparse, des rollouts trop courts ou une politique uniforme peuvent sous-estimer des actions pourtant stratégiquement bonnes.


### Exercice 3 — Sensibilité à $\kappa$

Sur le Blocking Maze, comparez les récompenses finales sur cinq graines pour $\kappa \in \{0,10^{-4},10^{-3},5\times10^{-3},10^{-2},10^{-1}\}$. Tracez les valeurs non nulles en échelle logarithmique et représentez $\kappa=0$ séparément.

**Indice** : réutilisez `RunBlocking` avec un budget identique pour chaque valeur. Décrivez le compromis observé au lieu de supposer quel réglage gagne.


In [12]:
// Exercice 3 — balayage du bonus kappa
var kappas = new[] { 0.0, 1e-4, 1e-3, 5e-3, 1e-2, 1e-1 };
var finalRewards = new List<double>(); // TODO étudiant : exécuter les 5 graines pour chaque kappa.
if (finalRewards.Count == 0)
{
    Console.WriteLine($"Exercice 3 à compléter : {kappas.Length} réglages définis, aucune récompense mesurée.");
}
else
{
    if (finalRewards.Count != kappas.Length || finalRewards.Any(value => !double.IsFinite(value) || value < 0))
        throw new InvalidOperationException("Il faut une récompense finale finie et non négative par valeur de kappa.");
    var bestIndex = finalRewards.IndexOf(finalRewards.Max());
    Console.WriteLine($"Contrôle kappa : {finalRewards.Count} mesures valides ; meilleur résultat observé κ={kappas[bestIndex]:G} avec {finalRewards[bestIndex]:F1} récompenses.");
}


Exercice 3 à compléter : 6 réglages définis, aucune récompense mesurée.


## Les cinq notions à retenir

| Concept | Idée clé |
|---|---|
| Modèle du monde | Mémoriser ou estimer la conséquence d’une action |
| Dyna-Q | Q-Learning réel et mises à jour simulées partagent la même règle |
| Efficacité d’échantillonnage | Le calcul peut économiser des interactions réelles |
| Modèle périmé | Un changement peut rendre le plan appris trompeur ; Dyna-Q+ réexplore |
| Decision-time planning | Rollouts évaluent des actions à l’état courant ; MCTS concentre les simulations |

## À retenir

- L’oracle BFS, le nombre de paires mémorisées et les courbes ancrent respectivement la géométrie, la couverture du modèle et l’apprentissage dans des mesures exécutées.
- Le nombre de mises à jour simulées augmente le calcul ; comparez les courbes et le seuil d’épisode réellement obtenus, sans confondre calcul et expérience réelle.
- Sur un monde changeant, observez les deux séries de récompenses et la transition de carte. Le bonus de récence est un mécanisme d’exploration, dont l’effet dépend du réglage et de la graine.
- Un rollout n’utilise pas de table Q mais dépend du modèle et de sa politique de simulation.

## Vérifiez votre compréhension

**1. Pourquoi le modèle ne contient-il pas immédiatement toutes les transitions de la grille ?**

<details><summary>Réponse</summary>Il n’enregistre que les paires (état, action) rencontrées. La planification uniforme ne peut échantillonner que ces paires mémorisées ; elle ne rend pas le modèle complet par elle-même.</details>

**2. Que change le paramètre n dans Dyna-Q ?**

<details><summary>Réponse</summary>Il fixe le nombre de mises à jour simulées après chaque transition réelle. n=0 laisse la boucle model-free ; augmenter n dépense davantage de calcul pour les mêmes interactions.</details>

**3. Pourquoi ajouter un bonus de récence quand l’environnement évolue ?**

<details><summary>Réponse</summary>Une action longtemps non revisitée peut révéler qu’une ancienne prédiction n’est plus valide. Le bonus rend certaines transitions anciennes ou inconnues plus attirantes lors de la planification.</details>

**4. Quelle limite a le rollout aléatoire ?**

<details><summary>Réponse</summary>Il dépend de la qualité du modèle, de la profondeur et de la politique de simulation. Avec une récompense éparse, les trajectoires aléatoires peuvent rarement atteindre le but et produire des estimations faibles.</details>

## Pour aller plus loin

- [RL-09 — RL offline : Behavior Cloning](RL-09-RL-Offline-Behavior-Cloning-Python.ipynb) : exploiter des données sans interaction en ligne.
- [GameTheory-17 — apprentissage multi-agent](../GameTheory/GameTheory-17-MultiAgent-RL-Python.ipynb) : MCTS et décisions dans des jeux à deux joueurs.
- [RL-21 — auto-jeu AlphaZero](RL-21-Auto-Jeu-AlphaZero-Python.ipynb) : recherche d’arbre guidée par réseau.

### Lectures

- Sutton & Barto, *Reinforcement Learning: An Introduction*, 2e éd., chapitre 8.
- Sutton (1990), *Integrated Architectures for Learning, Planning, and Reacting*.
- Moore & Atkeson (1993), *Prioritized Sweeping*.
- Silver et al. (2018), *AlphaZero* ; Schrittwieser et al. (2020), *MuZero*.
